In [2]:
import pdfplumber
import pandas as pd
import json
import numpy as np
from langchain_community.vectorstores import FAISS


from langchain_huggingface import HuggingFaceEmbeddings


from transformers import pipeline
from langchain_core.documents import Document


In [3]:
pdf_path = "sample_honeywell_report.pdf"

In [4]:
def extract_pdf(pdf_path):

    documents = []

    chunk_id = 0

    with pdfplumber.open(pdf_path) as pdf:

        for page_number, page in enumerate(pdf.pages):

            text = page.extract_text()

            if text:
                documents.append({
                    "page": page_number + 1,
                    "text": text,
                    "chunk_id": chunk_id
                })

                chunk_id += 1

            tables = page.extract_tables()

            for table in tables:

                df = pd.DataFrame(table)

                table_text = df.to_string(index=False)

                documents.append({
                    "page": page_number + 1,
                    "text": table_text,
                    "chunk_id": chunk_id
                })

                chunk_id += 1

    return documents

In [5]:
docs = extract_pdf(pdf_path)
docs

[{'page': 1,
  'text': 'Honeywell Avionics — Spares Supply Chain Report\nThis internal report summarises recent observations and KPIs for the spares supply chain for the Avionics\ndivision. It contains quantitative metrics, operational notes, and a sample parts table to support a\nRetrieval-Augmented-Generation (RAG) pipeline exercise.\nExecutive Summary\nOver the past quarter, lead times have varied across regions. Root causes include vendor delays and\ncustoms clearance. Recommendations include incremental automation for inventory reordering and\nimproved vendor SLAs.',
  'chunk_id': 0},
 {'page': 2,
  'text': 'Operational Observations\n(cid:127) Region A experienced a 12% increase in expedited shipments due to stockouts.\n(cid:127) Region B showed a 7-day increase in average lead time during monsoon months.\n(cid:127) Spares accuracy (picks vs orders) remained at 98.2% but returns increased slightly due to labeling\nerrors.\nNotes on data sources: Data aggregated from ERP, WMS, and 

In [6]:
def chunk_text(documents, chunk_size=300, overlap=50):

    chunks = []

    chunk_id = 0

    for doc in documents:

        words = doc["text"].split()

        start = 0

        while start < len(words):

            end = start + chunk_size

            chunk = " ".join(words[start:end])

            chunks.append({

                "chunk_id": chunk_id,

                "page": doc["page"],

                "text": chunk

            })

            chunk_id += 1

            start += chunk_size - overlap

    return chunks

In [7]:
chunks = chunk_text(docs)
print(len(chunks))
print(chunks[0])

8
{'chunk_id': 0, 'page': 1, 'text': 'Honeywell Avionics — Spares Supply Chain Report This internal report summarises recent observations and KPIs for the spares supply chain for the Avionics division. It contains quantitative metrics, operational notes, and a sample parts table to support a Retrieval-Augmented-Generation (RAG) pipeline exercise. Executive Summary Over the past quarter, lead times have varied across regions. Root causes include vendor delays and customs clearance. Recommendations include incremental automation for inventory reordering and improved vendor SLAs.'}


In [8]:
print(chunks[5])

{'chunk_id': 5, 'page': 4, 'text': '0 1 2 3 4 Part ID Description Category Qty On Hand Lead Time (days) P-1001 Avionics Control Module Control 25 30 P-1002 Navigation Sensor Sensor 10 45 P-1003 Hydraulic Actuator Actuator 5 60 P-1004 Power Regulator Power 40 20 P-1005 Comms Antenna Antenna 18 25'}


In [9]:
print(chunks[7])

{'chunk_id': 7, 'page': 6, 'text': 'Appendix Glossary (cid:127) ERP: Enterprise Resource Planning system. (cid:127) WMS: Warehouse Management System. (cid:127) SLA: Service Level Agreement. (cid:127) Lead Time: Time between order and receipt. Contact: Supply Chain Analytics Team — analytics@honeywell.example'}


In [10]:

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7383.45it/s]


In [11]:

documents = []

for chunk in chunks:
    documents.append(
        Document(
            page_content=chunk["text"],
            metadata={
                "page": chunk["page"],
                "chunk_id": chunk["chunk_id"]
            }
        )
    )

print(documents[0])

page_content='Honeywell Avionics — Spares Supply Chain Report This internal report summarises recent observations and KPIs for the spares supply chain for the Avionics division. It contains quantitative metrics, operational notes, and a sample parts table to support a Retrieval-Augmented-Generation (RAG) pipeline exercise. Executive Summary Over the past quarter, lead times have varied across regions. Root causes include vendor delays and customs clearance. Recommendations include incremental automation for inventory reordering and improved vendor SLAs.' metadata={'page': 1, 'chunk_id': 0}


In [12]:

from langchain_community.vectorstores import FAISS
vector_db = FAISS.from_documents(
    documents,
    embedding_model
)

vector_db.save_local("vector_db")

In [13]:
vector_db